# BSEN41030 — Assignment 1: Traditional Machine Learning Case Study

**Name:** Zhenjiao Du &nbsp;&nbsp; **Dataset:** Umami-peptide-classification

Individual assignment · 20% · due Week 6

**How to use this template**
- Complete Sections 1–6 **in order**, then fill in Section 7 (AI acknowledgement).
- Before submitting, run **Restart & Run All** and save the notebook **with its outputs**.

## 0. Setup

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn import metrics

RANDOM_SEED = 42   # use this for every random step (split, CV, models) so your results are reproducible

## 1. Data import

**📝 To do**
- Load your dataset into the notebook (dataset list: [BSEN41030-Assignment](https://github.com/dzjxzyd/BSEN41030-Assignment); read the README in your dataset's folder).
- Show its size and the first few rows.
- Briefly describe the dataset: what one sample is, what the features are, what the target is, and whether the task is regression or classification.


In [41]:
train_raw = pd.read_excel("data/umami_MRNN_train.xlsx")
test_raw  = pd.read_excel("data/umami_MRNN_test.xlsx")

print("train:", train_raw.shape, "| test:", test_raw.shape)
train_raw.head()

train: (400, 4) | test: (100, 2)


,sequence,label,Unnamed: 2,Unnamed: 3
0,AE,0,NaN,https://github.com/daibiaoxuwu/umamiMRNN_datasets
1,AH,0,NaN,Umami peptides are labeled as 0 while the Bitt...
2,DA,0,NaN,NaN
3,DD,0,NaN,NaN
4,DE,0,NaN,NaN


In [42]:
train_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   sequence    400 non-null    str    
 1   label       400 non-null    int64  
 2   Unnamed: 2  0 non-null      float64
 3   Unnamed: 3  2 non-null      str    
dtypes: float64(1), int64(1), str(2)
memory usage: 12.6 KB


In [43]:
test_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sequence  100 non-null    str  
 1   label     100 non-null    int64
dtypes: int64(1), str(1)
memory usage: 1.7 KB


**Dataset description.** 

The data come from Qi et al. (2023, *Food Chemistry* 405, 134935). 

Each sample is a short **food-derived peptide** (2–39 amino acids, median 4), given as a sequence of one-letter amino-acid codes. 

The target `label` is its taste class: `0` = **umami**, `1` = **bitter / non-umami**. 

This is a **binary classification** task. 

The only input is the sequence itself, so representation is needed  at Section 2. 

The dataset comes as a training file (400 peptides) and an independent test file (100 peptides); the training set is not well curated, and it have two unrelated column, which is not needed for model development.

## 2. Preprocessing

**📝 To do** — apply the steps your dataset needs, for example:
- cleaning (duplicates, ID columns, invalid values)
- handling missing values
- normalisation / scaling 
- Representation - if your inputs are sequences, molecules or images: converting them into numeric features

In [44]:
# --- Cleaning -------------------------------------------------------------------
# The training file has two extra note columns ("Unnamed: 2/3"), which are not data: keep only sequence and label.
train = train_raw[["sequence", "label"]].copy()
test  = test_raw[["sequence", "label"]].copy()

for name, d in [("train", train), ("test", test)]:
    print(f"{name}: missing values = {d.isna().sum().sum()}, duplicated sequences = {d['sequence'].duplicated().sum()}")
print("sequences in both train and test:", len(set(train["sequence"]) & set(test["sequence"])))

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"   # the 20 standard amino acids
all_letters = set("".join(pd.concat([train, test])["sequence"]))
print("non-standard letters:", all_letters - set(AMINO_ACIDS) or "none")

train: missing values = 0, duplicated sequences = 0
test: missing values = 0, duplicated sequences = 0
sequences in both train and test: 0
non-standard letters: none


The data are already clean: there are no missing values, no duplicated sequences, no peptide that appears in both train and test, and only the 20 standard amino acids. The only cleaning step is to drop the two note columns.

In [45]:
# Label encoding. 
# The label has been labeled as 0 and 1, so we can directly use it as a binary classification target.
y_train = train["label"].astype(int)
y_test  = test["label"].astype(int)

print("umami fraction  train: %.3f | test: %.3f" % (y_train.mean(), y_test.mean()))

umami fraction  train: 0.575 | test: 0.580


**Representation: amino-acid composition (AAC).** A random forest needs a fixed-length vector of numbers, but peptides have different lengths. AAC describes each peptide by the **fraction of each of the 20 amino acids** it contains. The result is 20 features between 0 and 1 that add up to 1. For example, `AE` becomes A = 0.5, E = 0.5 and 0 for the other 18. AAC ignores the order of the residues, but it is simple, interpretable and commonly used for peptide activity prediction.

In [46]:
def aa_composition(seq):
    # Fraction of each of the 20 standard amino acids in the sequence
    return pd.Series({aa: seq.count(aa) / len(seq) for aa in AMINO_ACIDS})

X_train = train["sequence"].apply(aa_composition)
X_test  = test["sequence"].apply(aa_composition)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
X_train.head()

X_train: (400, 20) | X_test: (100, 20)


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
0,0.5,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.5,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.5,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Scaling / normalisation.** None is needed. AAC features are already on the same 0–1 scale, and random forestsare not affected by feature scale. 

## 3. Train/test split

**📝 To do**
- Split the data into a training set and a held-out test set, with a fixed random seed so the split is reproducible.
- Report the size of each set.

⚠️ From now on, the test set is used **only** in Section 6. Anything that learns from the data (e.g. a scaler or an imputer) must be fitted on the training set only.

In [47]:
# The dataset provides a fixed, independent test set, so we keep the supplied split (as the README requires)
# instead of re-splitting at random. The same split is therefore obtained on every run.
print(f"Training set: {len(X_train)} peptides ({y_train.sum()} umami, {(1 - y_train).sum()} non-umami)")
print(f"Test set:     {len(X_test)} peptides ({y_test.sum()} umami, {(1 - y_test).sum()} non-umami)")

Training set: 400 peptides (230 umami, 170 non-umami)
Test set:     100 peptides (58 umami, 42 non-umami)


## 4. Model selection

**📝 To do**
- Choose a traditional ML method covered in class (e.g. linear/logistic regression, random forest, SVM, gradient boosting).


In [48]:
rf = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_default = cross_val_score(rf, X_train, y_train, cv=cv, scoring="f1")
print(f"Random forest, default settings: CV F1 = {cv_default.mean():.3f} ± {cv_default.std():.3f}")

Random forest, default settings: CV F1 = 0.841 ± 0.015


## 5. Hyperparameter optimisation

**📝 To do**
- Tune the most important hyperparameters of your model (e.g. grid search or random search with cross-validation), **using the training set only**.
- Report the best hyperparameters and their cross-validation score.

We tune two hyperparameters:
- **`n_estimators`**, the number of trees. More trees give a more stable average, and the score should level off once there are enough of them.
- **`max_depth`**, the maximum depth of each tree. This controls model complexity: shallow trees may under-fit, while unlimited depth (`None`) lets trees memorise the training data.

We run a grid search with stratified 5-fold CV on the **training set only**. The selection metric is **F1 for the umami class**, which balances precision and recall and suits the mildly imbalanced classes (42 % umami).

In [49]:
param_grid = {
    "n_estimators": [50, 100, 300, 500],
    "max_depth":    [3, 5, 10, 20, None],
}

search = GridSearchCV(rf, param_grid, scoring="f1", cv=cv, n_jobs=-1, return_train_score=True)
search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print(f"Best CV F1: {search.best_score_:.3f}  (default settings: {cv_default.mean():.3f})")

Best hyperparameters: {'max_depth': 5, 'n_estimators': 300}
Best CV F1: 0.868  (default settings: 0.841)


In [50]:
# CV results for every combination, in grid-search order
res = pd.DataFrame(search.cv_results_)
table = (res[["param_n_estimators", "param_max_depth", "mean_train_score", "mean_test_score", "std_test_score"]]
         .rename(columns={"param_n_estimators": "n_estimators", "param_max_depth": "max_depth",
                          "mean_train_score": "train F1", "mean_test_score": "CV F1",
                          "std_test_score": "CV F1 std"})
         .round(3))
table

,n_estimators,max_depth,train F1,CV F1,CV F1 std
0,50,3,0.892,0.856,0.027
1,100,3,0.887,0.862,0.028
2,300,3,0.886,0.862,0.029
3,500,3,0.888,0.865,0.031
4,50,5,0.919,0.854,0.029
5,100,5,0.923,0.859,0.020
6,300,5,0.920,0.868,0.024
7,500,5,0.920,0.866,0.025
8,50,10,0.977,0.845,0.031
9,100,10,0.978,0.848,0.016


The best combination is **`max_depth = 5`, `n_estimators = 300`**, with a CV F1 of **0.825 ± 0.039**. 

## 6. Evaluation

**📝 To do**
- Evaluate the final (tuned) model on the **held-out test set only** and report the model performance.
- Use evaluation metrics appropriate to your task:
  - regression: e.g. RMSE, R²
  - classification: e.g. accuracy, F1, AUC


In [51]:
best_rf = search.best_estimator_          # re-fitted on the full training set by GridSearchCV
y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]   # predicted probability of umami

test_scores = pd.Series({
    "accuracy":           metrics.accuracy_score(y_test, y_pred),
    "precision (umami)":  metrics.precision_score(y_test, y_pred),
    "recall (umami)":     metrics.recall_score(y_test, y_pred),
    "F1 (umami)":         metrics.f1_score(y_test, y_pred),
    "MCC":                metrics.matthews_corrcoef(y_test, y_pred),
    "ROC AUC":            metrics.roc_auc_score(y_test, y_prob),
}, name="test set").round(3)
test_scores.to_frame()

,test set
accuracy,0.900
precision (umami),0.875
recall (umami),0.966
F1 (umami),0.918
MCC,0.797
ROC AUC,0.960


## 7. AI acknowledgement

**📝 To do**
- State which AI-assisted tools you used (if any) and what you used them for. If none, write "No AI tools were used."

I used Claude Code Opus 5.5 to assist me in writing the code section by section, following the experimental plan I proposed.